In [3]:
import sys
sys.path.append('../')

from ingestion import load_faq_data, build_index
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [4]:
documents = load_faq_data()
index = build_index(documents)


Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 402 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 79 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 255 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 472 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1342
Building index with 1342 documents
Text fields: ['question', 'sectio

In [5]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [7]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [10]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [11]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [12]:
result.cost

CostInfo(input_cost=Decimal('0.00157125'), output_cost=Decimal('0.0015705'), total_cost=Decimal('0.00314175'))

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run install setup local model server FAQ"}', call_id='call_7GcknizIcXbwofRa2Ycwu5Kj', name='search', type='function_call', id='fc_0b6be02052a42645006a2b2e210da4819cbbca6e6762302128', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Ollama local run install macOS Windows Lin

In [14]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [15]:
runner.run()


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='I do I run Olama', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama run Ollama how to run install start service command line"}', call_id='call_sfj5elCBTcAvFeOfdJkM1Qrs', name='search', type='function_call', id='fc_08d30938660339b0006a2b307112908192969c34b64b0f11ea', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"Ollama getting started r